# 04 - Validators and the hand-check sheet

Runs the checks that try to catch Gemini's mistakes (Jev, TypeSafe's yes/no checker, repeated runs for
consistency, agreement with the classic models, cleanlab label-noise detection) and builds the Google
Sheet you will review by hand. **Spends money** (Jev validator, summary judge, two rerun passes): roughly
$1-2 in total, estimated below before anything is spent. Rerunnable; finished steps are cached.

**Step 1 - setup (free).** Downloads the project code from GitHub, installs it, connects Google Drive
(all results are saved in `MyDrive/cfpb-nlp-classifier/phase1`), reads your `GEMINI_API_KEY` secret (and the optional `TYPESAFE_API_KEY` Jev secret) and
logs in to Google Sheets. Colab will ask you to allow Drive and Sheets access - click Allow.
Takes 2-4 minutes the first time. If it says the key is missing: click the key icon on the left of
Colab, add a secret named `GEMINI_API_KEY`, and switch "Notebook access" on.
This notebook's Jev step needs the `TYPESAFE_API_KEY` secret (same place, Notebook access on).

In [ ]:
!git clone -q https://github.com/Dentoneer/cfpb-nlp-classifier /content/cfpb-nlp-classifier || git -C /content/cfpb-nlp-classifier pull -q
%pip install -q -e "/content/cfpb-nlp-classifier[ml,colab]"
import sys; sys.path.insert(0, "/content/cfpb-nlp-classifier")
from nlp.colab import setup
cfg, gc, api_key, jev_key = setup()

**Load saved results and tools (free).**

In [ ]:
from nlp import colab, store, pipeline as P
from nlp.classifiers.tfidf import TfidfLR
from nlp.llm.client import GenAIClient
from nlp.llm.jev import HttpJevClient
from nlp.review import review_rows, select_hand_check
from nlp.sheets import write_review_sheet
complaints = store.read(cfg, P.STAGE_COMPLAINTS)
taxonomy = store.read(cfg, P.STAGE_TAXONOMY)
splits = store.read(cfg, P.STAGE_SPLITS)
gem = store.read(cfg, P.STAGE_PRED_GEMINI)
train_df, test_df = P.train_test_frames(complaints, splits)
sc_rows = P.split_rows(complaints, splits, splits["self_consistency"])
ledger, gem_cache, judge_cache, limiter = colab.open_tools(cfg)
client = GenAIClient(api_key)
jev_client = HttpJevClient(jev_key)

**Estimate (free).** Cost of each step below (measured from what 03 really spent, plus the worst-case
bound). Each paid step checks its own estimate against the remaining cap before it starts.

In [ ]:
est = colab.phase_estimate(cfg, ledger, taxonomy, test_df, sc_rows)


def gate(stage, label):
    colab.check_fits(ledger, colab.pending_usd(est, [stage]), label,
                     bound_usd=colab.pending_usd(est, [stage], "bound_usd"))

**Self-consistency (paid, ~$0.1-0.4).** Re-asks Gemini about 150 test complaints twice more. If its answer
changes between runs, it is probably unsure. Look at: the spend line.

In [ ]:
ids = sc_rows["complaint_id"]
try:
    gate(P.STAGE_PRED_GEMINI_SC2, "Self-consistency run 2")
    sc2 = colab.stage_paid(cfg, P.STAGE_PRED_GEMINI_SC2, lambda: P.run_gemini(
        cfg, sc_rows, client, ledger, gem_cache, taxonomy, "gemini_sc2", limiter), ids, ledger)
    gate(P.STAGE_PRED_GEMINI_SC3, "Self-consistency run 3")
    sc3 = colab.stage_paid(cfg, P.STAGE_PRED_GEMINI_SC3, lambda: P.run_gemini(
        cfg, sc_rows, client, ledger, gem_cache, taxonomy, "gemini_sc3", limiter), ids, ledger)
finally:
    print(colab.ledger_report(ledger))

**Jev validator (paid, about $0.03).** Jev (TypeSafe AI) reads each complaint plus Gemini's chosen
Product/Issue/Sub-issue and gives a probability that each level is right (one request per complaint).
Runs on all 800 test complaints.

In [ ]:
try:
    gate(P.STAGE_JEV, "Jev validator")
    jev = colab.stage_paid(cfg, P.STAGE_JEV, lambda: P.run_jev(
        cfg, gem, test_df, taxonomy, jev_client, ledger, judge_cache, limiter), test_df["complaint_id"], ledger)
finally:
    print(colab.ledger_report(ledger))

**Pick the hand-check complaints, then judge their summaries (paid, ~$0.05).** Chooses about 200 test
complaints, half where Gemini agrees with CFPB and half where it does not, and has the judge grade
Gemini's one-paragraph summaries for those.

In [ ]:
selection = store.stage(cfg, P.STAGE_HAND_CHECK, lambda: select_hand_check(
    gem, complaints, cfg.hand_check_n, cfg.seed))
try:
    gate(P.STAGE_SUMMARY_JUDGE, "Summary judge")
    flags = colab.stage_paid(cfg, P.STAGE_SUMMARY_JUDGE, lambda: P.run_summary_judge(
        cfg, gem, test_df, selection, client, ledger, judge_cache, limiter),
        selection["complaint_id"], ledger)
finally:
    print(colab.ledger_report(ledger))

**Free validators (a few minutes, no cost).** Agreement with CFPB labels, agreement with the classic
models, Gemini's own confidence, self-consistency, a hierarchy check, and cleanlab (finds labels that
look wrong, using 5-fold out-of-fold TF-IDF probabilities on the training set).

In [ ]:
probs = {n: store.read(cfg, P.stage_probs(n)) for n in ["tfidf", "embed_lr", "knn", "modernbert"]
         if store.exists(cfg, P.stage_probs(n))}
clean = store.stage(cfg, P.STAGE_CLEANLAB, lambda: P.cleanlab_validator(
    cfg, train_df, test_df, gem, probs["tfidf"], TfidfLR, 5))
vres = store.stage(cfg, P.STAGE_VALIDATORS, lambda: P.build_validator_results(
    complaints, taxonomy, gem, [sc2, sc3], probs, clean, jev))
print(vres.groupby("validator").size())

**Create the review sheet (free).** Writes a Google Sheet with each complaint, the CFPB labels, Gemini's
answer and summary. Where Gemini and CFPB agree the verdict is pre-filled `both OK`; the rest are blank
dropdowns for you. **Look at:** the printed link.

In [ ]:
rows = review_rows(selection, gem, complaints)
url = colab.publish_once(cfg, "review_sheet", lambda: write_review_sheet(gc, rows, "CFPB hand check"))
print("REVIEW SHEET:", url)
print(colab.ledger_report(ledger))

# STOP - your turn

**Open the review sheet (URL printed above), fill the dropdowns, then run notebook 05.**

For each row pick a verdict per level (`model right`, `label right`, `both OK`, `both wrong`, or
`ambiguous` if you cannot tell) and mark whether Gemini's summary is OK (`yes` / `no`). Leave no blank
verdict cells.